# Agribusiness Data Analysis Planning - Week 1
## Notebook 02: Data Acquisition Architecture, Ingestion Protocols & Provenance

### Purpose
This notebook tests and demonstrates the modular data acquisition pipeline designed in `src/data_collection.py`. It establishes:
1. Automated API querying with retry logic
2. Schema enforcement and column contract validation
3. Cryptographic SHA-256 checksum generation for immutable audit logging

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.data_collection import WorldBankAgriCollector, LocalDatasetIngestor, inspect_raw_dataset
from src.utils import calculate_sha256

raw_dir = repo_root / "data" / "raw"
print("Configured Raw Data Directory:", raw_dir)

### 1. Ingestion Pipeline Execution: World Bank Macro Indicators
We fetch official cereal yield (`AG.YLD.CREL.KG`) and agricultural land percentage (`AG.LND.AGRI.ZS`) time series for India and the World.

In [ ]:
wb_collector = WorldBankAgriCollector(raw_dir=raw_dir)

# Attempt live ingestion of Cereal Yield for India
df_yield = wb_collector.fetch_indicator(
    country="IND",
    indicator_key="cereal_yield",
    date_range="1990:2022"
)

if df_yield is not None:
    print(f"Ingested {len(df_yield)} records. First 5 rows:")
    display(df_yield.head())
else:
    print("World Bank API query bypassed or unavailable. Pipeline ready for Week 2 ingestion batch.")

### 2. Schema Contract & Ingestion Profiling Demonstration
We verify that our schema inspector accurately profiles row counts, data types, missing cells, and duplicate records.

In [ ]:
# Demonstration of schema validation logic on synthetic audit frame
mock_audit_df = pd.DataFrame({
    "state_name": ["Punjab", "Haryana", "Uttar Pradesh", "Punjab"],
    "district_name": ["Ludhiana", "Karnal", "Meerut", "Ludhiana"],
    "crop_year": [2020, 2020, 2020, 2020],
    "season": ["Rabi", "Rabi", "Rabi", "Rabi"],
    "crop": ["Wheat", "Wheat", "Wheat", "Wheat"],
    "area": [250000.0, 180000.0, 310000.0, 250000.0],
    "production": [1200000.0, 860000.0, 1420000.0, 1200000.0]
})

diagnostics = inspect_raw_dataset(mock_audit_df, dataset_name="APY_Schema_Validation_Test")
print("Diagnostic profiling summary:")
for k, v in diagnostics.items():
    print(f"  {k}: {v}")

### 3. Data Integrity & Audit Manifest Check
Reviewing the audit log created in `data/raw/manifests/audit_log.json` to verify cryptographic hashes and provenance tracking.

In [ ]:
manifest_file = raw_dir / "manifests" / "audit_log.json"
if manifest_file.exists():
    import json
    with open(manifest_file, "r") as f:
        manifest = json.load(f)
    print("Audit Manifest Entries:", len(manifest))
    print(json.dumps(manifest, indent=2))
else:
    print("Audit manifest will be created during Week 2 batch download.")